In [19]:
import pandas as pd
import numpy as np
import re

crosswalk = pd.read_excel('../data/raw/TRACT_ZIP_122022.xlsx')

def decode_excel_xml(s):
    return re.sub(r'_x([0-9A-Fa-f]{4})_', 
                  lambda m: chr(int(m.group(1), 16)), str(s))

crosswalk['TRACT'] = crosswalk['TRACT'].apply(decode_excel_xml)
crosswalk['ZIP']   = crosswalk['ZIP'].apply(decode_excel_xml)


print("after decode TRACT:")
print(crosswalk['TRACT'].head(5).tolist())
print()
print("after decode ZIP:")
print(crosswalk['ZIP'].head(5).tolist())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


after decode TRACT:
['01001020100', '01001020200', '01001020300', '01001020400', '01001020400']

after decode ZIP:
['36067', '36067', '36067', '36067', '36066']


In [24]:
noise = pd.read_csv(
    '../data/ca_tract_noise.csv'
)
noise = noise.rename(columns={'GEOID': 'tract_id', 'mean_noise': 'noise_mean_db'
                              ,'max_noise': 'noise_max_db', 'min_noise': 'noise_min_db'})
print(noise.head())
print(noise.shape)

     tract_id    NAME  noise_mean_db  noise_max_db  noise_min_db
0  6001442700  4427.0      60.030842     86.239731     45.016068
1  6001442800  4428.0      60.857664     85.595474     45.048786
2  6037204920  2049.2      57.251847     87.710724     45.036545
3  6037205110  2051.1      55.101172     73.636124     45.219223
4  6037205120  2051.2      56.199094     76.095947     45.003738
(9092, 5)


In [25]:

crosswalk_sd = crosswalk.copy()
crosswalk_sd = crosswalk_sd.rename(columns={'TRACT': 'tract_id', 'ZIP': 'zcta'})

print("San Diego num:", len(crosswalk_sd))
print("TRACT sample:", crosswalk_sd['tract_id'].head(3).tolist())
print("ZIP sample:", crosswalk_sd['zcta'].head(3).tolist())

San Diego num: 172154
TRACT sample: ['01001020100', '01001020200', '01001020300']
ZIP sample: ['36067', '36067', '36067']


In [26]:

crosswalk_sd['tract_id'] = crosswalk_sd['tract_id'].astype('int64')


noise_mapped = pd.merge(noise, crosswalk_sd[['tract_id', 'zcta', 'RES_RATIO']], 
                         on='tract_id', how='left')

print("merge num :", len(noise_mapped))
print("have't match tract:", noise_mapped['zcta'].isna().sum())
print()
print(noise_mapped.head(3).to_string())

merge num : 14319
have't match tract: 2247

     tract_id    NAME  noise_mean_db  noise_max_db  noise_min_db   zcta  RES_RATIO
0  6001442700  4427.0      60.030842     86.239731     45.016068  94536        1.0
1  6001442700  4427.0      60.030842     86.239731     45.016068  94538        0.0
2  6001442800  4428.0      60.857664     85.595474     45.048786  94538        1.0


In [27]:
def weighted_noise(g):
    total_weight = g['RES_RATIO'].sum()
    if total_weight == 0:
        return pd.Series({
            'noise_mean_db': g['noise_mean_db'].mean(),
            'noise_max_db':  g['noise_max_db'].mean(),
            'noise_min_db':  g['noise_min_db'].mean(),
        })
    return pd.Series({
        'noise_mean_db': np.average(g['noise_mean_db'], weights=g['RES_RATIO']),
        'noise_max_db':  np.average(g['noise_max_db'],  weights=g['RES_RATIO']),
        'noise_min_db':  np.average(g['noise_min_db'],  weights=g['RES_RATIO']),
    })

noise_zcta = (
    noise_mapped.dropna(subset=['zcta']) 
    .groupby('zcta')
    .apply(weighted_noise)
    .reset_index()
)

print("ZCTA:", len(noise_zcta))
print("zcta sample:", noise_zcta['zcta'].head(5).tolist())
print(noise_zcta.head(5).to_string())
noise_zcta.to_csv('../data/processed/zcta/zcta_noise.csv', index=False)
print("✅ ")

ZCTA: 2183
zcta sample: ['90001', '90002', '90003', '90004', '90005']
    zcta  noise_mean_db  noise_max_db  noise_min_db
0  90001      55.280064     74.542302     45.102884
1  90002      55.551904     74.367805     45.061863
2  90003      57.612337     83.350455     45.052543
3  90004      56.249286     74.941374     45.100273
4  90005      54.854745     69.565419     45.163183
✅ 


/var/folders/lc/n2nwqh494p91vlgh3_xnxj080000gp/T/ipykernel_21448/929952974.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_noise)


In [28]:
# read all
socioeconomic = pd.read_csv('../data/processed/zcta/zcta_demographics_selected.csv')
demographics  = pd.read_csv('../data/processed/zcta/zcta_demographics.csv')
housing       = pd.read_csv('../data/processed/zcta/zcta_housing.csv')
housing_sd    = housing[housing['State'] == 'CA'].rename(columns={'zip_code': 'zcta'})

noise_zcta['zcta']    = noise_zcta['zcta'].astype(str).str.zfill(5)
socioeconomic['zcta'] = socioeconomic['zcta'].astype(str).str.zfill(5)
demographics['zcta']  = demographics['zcta'].astype(str).str.zfill(5)
housing_sd['zcta']    = housing_sd['zcta'].astype(str).str.zfill(5)

# Merge
master = noise_zcta.copy()
master = pd.merge(master, socioeconomic, on='zcta', how='left')
master = pd.merge(master, demographics,  on='zcta', how='left')
master = pd.merge(master, housing_sd[['zcta', 'home_value']], on='zcta', how='left')

master['poverty_rate'] = master['poverty_rate_x'].combine_first(master['poverty_rate_y'])
master = master.drop(columns=['poverty_rate_x', 'poverty_rate_y'])

print("Shape:", master.shape)
print("\nmissing:")
print(master.isnull().sum())

master.to_csv('../data/processed/master_dataset.csv', index=False)
print("\ncomplete")

Shape: (2183, 17)

missing:
zcta                         0
noise_mean_db                0
noise_max_db                 0
noise_min_db                 0
median_household_income    680
unemployment_rate          531
population                 671
pnhwhite                   671
pnhblack                   671
phispanic                  671
pforeign_born              671
punemployed                671
affluence                  671
disadvantage               671
median_family_income       671
home_value                 710
poverty_rate               575
dtype: int64

complete
